# The App

Book 3 could rank papers. Give it a query, it gave back a sensible ordering —
and that was as far as it went. It printed titles into an output area, and the
next query started from exactly the same place as the last one. Nothing you did
with the results changed anything.

This book turns that ranking into something you can *use* and something that
*learns*: five papers as cards you can click, three ways to react to each one,
and a next question that is ranked differently because of what you pressed.
Every search and every click lands on disk in a shape book 5 can score.

**Five papers on a screen.** Book 3's output was a line of text per paper — a
score and a title. That's enough to check whether search works, and not enough
to decide whether *you* want the paper: no abstract, no date, no way to open it,
and nowhere to say "yes, that one". A card is the same result with room for all
of that. `render_papers` draws one block per paper — linked title, the score
beside it, authors and date, then the abstract — and stacks them in a column.
Everything below builds on the store from book 3, unchanged, and every
embedding it needs is already in the cache, so this cell makes no API call.

In [1]:
from readnext.config import PAPERS_FILE
from readnext.corpus import chunk, load
from readnext.embed import embed_texts
from readnext.search import build_dense_index, search
from readnext.ui import render_papers

papers = load(PAPERS_FILE)
chunks = chunk(papers, strategy="small_to_big")
vectors = embed_texts([c.text for c in chunks])   # all cached — nothing reaches the API
store = build_dense_index(chunks, vectors)
by_id = {p.id: p for p in papers}

hits = search(store, "papers on fine-tuning", k=5)
render_papers([by_id[hit.paper_id] for hit in hits], [hit.score for hit in hits])

**A button is a callback.** Those cards are a picture — there is nothing to
press. The thing that changes that is a *widget*: a Python object that draws
itself in the notebook's output area and keeps a list of functions to call when
someone interacts with it. `button.on_click(f)` adds `f` to that list; when the
button is pressed, the browser tells the kernel, and the kernel runs `f`,
passing it the button that was clicked.

The part worth sitting with is what *doesn't* happen: no cell is re-run. The
cell that created the button finished long ago. The kernel is idle, waiting, and
your handler runs on its own — so anything it changes (a list, a profile, the
contents of another widget) stays changed, and the next click sees it. That is
the whole mechanism behind the rest of this book.

Here is the smallest version of it: one button, one handler that appends to a
list. `Output` is a widget that catches whatever is printed inside its `with`
block and displays it in place, which is how the handler can show you something
without a cell of its own.

In [2]:
import ipywidgets as widgets

clicks = []
button = widgets.Button(description="click me")
log = widgets.Output()

def on_click(clicked_button):
    clicks.append(len(clicks) + 1)
    with log:
        print(f"click {clicks[-1]} — the list now holds {clicks}")

button.on_click(on_click)
widgets.VBox([button, log])

**Three things a click can mean.** A card offers three responses. 👍 *like* —
the abstract appealed to me. 📖 *reading it* — I'm actually going to spend time
on this one. 👎 *dislike* — no, not this.

It's tempting to read 📖 as a stronger 👍 and weigh it more heavily. Resist that
for now: they are different kinds of evidence, not two points on one scale. A
like is a judgement about an abstract; "reading it" is a decision about your
afternoon, which is also affected by how long the paper is and how much time you
have. Which one better predicts what you'll click next is a question with a
number for an answer, and there is no number until book 5. So all three are
recorded separately — that keeps the option open — and the two positives count
the same today.

A `Profile` is those three lists of paper ids and nothing else. `record()` moves
a paper between them, so pressing 👎 on something you'd liked replaces the old
signal instead of leaving the paper in two lists at once.

In [3]:
from readnext.profile import Profile

demo = Profile(user="demo")
demo.record("2608.21702v1", "like")
demo.record("2608.16515v1", "reading")
demo.record("2607.26339v1", "dislike")
demo.record("2608.16515v1", "like")        # changed my mind — moves, not duplicates

demo

Profile(user='demo', liked=['2608.21702v1', '2608.16515v1'], disliked=['2607.26339v1'], reading=[])

**A taste vector is an average.** Each paper is a point in the embedding space —
1,536 numbers, and papers about similar things sit near each other. That was the
whole basis of book 3's search: put the *query* in the same space and look at
what's nearest.

So if two papers are the kind of thing you like, what single point stands for
"the kind of thing you like"? The obvious answer is the one in the middle of
them: average the two, coordinate by coordinate. With three-number toy vectors
instead of 1,536:

```
paper A   [0.9,  0.4,  0.1 ]
paper B   [0.7,  0.2,  0.6 ]
          ─────────────────
mean      [0.8,  0.3,  0.35]      (0.9+0.7)/2, (0.4+0.2)/2, (0.1+0.6)/2
```

One catch. Book 3's scores are cosine similarities, which only work as a
comparison because every vector in the store has length 1 — then the dot product
between two of them lands in a tidy, comparable range. The average of two
length-1 vectors is *shorter* than 1 (unless they point exactly the same way):
here its length is √(0.8² + 0.3² + 0.35²) = 0.92. Dividing every coordinate by
that length puts the point back on the unit sphere without changing the direction
it points in:

```
mean / 0.92   →   [0.866, 0.325, 0.379]
```

Direction is the part that carries the meaning; length would only make the
scores incomparable. `taste_vector()` does exactly this — average, then rescale.
The papers themselves are averaged from their chunks first, by `paper_vectors()`:
search works on chunks, but you click a whole paper, so a paper's vector is the
average of its chunks, rescaled the same way.

In [4]:
import numpy as np
from readnext.search import dedupe_to_papers, paper_vectors

vector_of = paper_vectors(chunks, vectors)

def nearest(vector, skip=(), k=5):
    """Titles of the k papers closest to `vector`, ignoring ones already judged."""
    hits = dedupe_to_papers(store.search(vector, k=200), k=200)
    kept = [hit for hit in hits if hit.paper_id not in skip][:k]
    return [(f"{hit.score:.3f}", by_id[hit.paper_id].title) for hit in kept]

liked = ["2608.21702v1", "2608.16515v1"]   # two papers on making RAG retrieval more precise
for paper_id in liked:
    print(by_id[paper_id].title)

olek = Profile(user="olek", liked=liked)
taste = olek.taste_vector(vector_of)

# the same arithmetic as the toy example, on the first 3 of 1,536 coordinates
mean = np.mean([vector_of[paper_id] for paper_id in liked], axis=0)
print(f"\nfirst 3 coordinates of each liked paper: {vector_of[liked[0]][:3]}  {vector_of[liked[1]][:3]}")
print(f"their mean:  {mean[:3]}   (length {np.linalg.norm(mean):.3f})")
print(f"rescaled:    {taste[:3]}   (length {np.linalg.norm(taste):.3f})")

print("\nnearest to that taste, papers I've already judged left out:")
for score, title in nearest(taste, skip=set(liked)):
    print(" ", score, title[:80])

From Association to Causation: Improving Retrieval Precision of Retrieval-Augmented Generation via Causal Relations and an Attention Mechanism
When Context Misleads: Intent-Guided Decoding for Robust Retrieval-Augmented Generation

first 3 coordinates of each liked paper: [-0.00642447  0.04007903  0.05295084]  [0.00959326 0.0193449  0.01098714]
their mean:  [0.0015844  0.02971197 0.03196899]   (length 0.891)
rescaled:    [0.00177752 0.03333357 0.03586571]   (length 1.000)

nearest to that taste, papers I've already judged left out:
  0.749 W-RAG: Source-Aware Retrieval for Enterprise Document Generation from Heterogene
  0.747 EnSI-RAG: Entity-Structure-Indexed Retrieval-Augmented Generation for Long-Docum
  0.738 Trustworthy RAG: An Evaluation Agent for Detecting Misinformation and Knowledge 
  0.733 GTA-RAG: Graph-Trajectory-Augmented Reinforcement Learning for Multi-Turn Retrie
  0.732 Bridging the Question-Answer Gap in Retrieval-Augmented Generation: Hypothetical


**Dislikes subtract.** The average above only knows what you wanted. A 👎 says
"not that", and the way to spend it is to build a second average — the middle of
everything you rejected — and *move away* from it. Not to it: subtract it.

How far away is a dial, called γ (gamma). Continuing the toy numbers, with one
rejected paper C = [0.1, 0.9, 0.2]:

```
mean of likes        [0.8,  0.3,   0.35]
γ = 0                [0.8,  0.3,   0.35]    dislikes ignored — the average above
γ = 0.5   − 0.5·C    [0.75, −0.15, 0.25]    pulled away
γ = 1     − 1·C      [0.7,  −0.6,  0.15]    dislike weighs as much as a like
```

Watch the middle coordinate. C's strongest direction was the second one, and
subtracting it turns your 0.3 there into −0.15: a paper that scores high in that
direction now scores *worse* than a paper with nothing there at all. That is the
whole point — a dislike doesn't merely stop promoting something, it demotes its
neighbours. Which is also why γ = 1 is risky: reject enough and the vector can
end up pointing away from your own interests. The default here is γ = 0.5, a
half-strength push, and book 5 is where it gets swept for a real value.

Below, one 👎 goes onto the profile: *"Trustworthy RAG: An Evaluation Agent for
Detecting Misinformation and Knowledge Poisoning in Generative AI Systems"*,
which is currently in the top five. **Predict before you run it:** of the other
four, how many survive into the new top five?

In [5]:
olek.record("2608.21095v1", "dislike")
print("disliked:", by_id["2608.21095v1"].title)

before = [title for _, title in nearest(taste, skip=set(liked))]
after = nearest(olek.taste_vector(vector_of, gamma=0.5), skip=set(liked) | {"2608.21095v1"})

print("\nnearest to the taste now:")
for score, title in after:
    print(" ", "kept" if title in before else "NEW ", score, title[:75])
print("\ndropped out:")
for title in before:
    if title not in [t for _, t in after] and title != by_id["2608.21095v1"].title:
        print("  ", title[:75])

disliked: Trustworthy RAG: An Evaluation Agent for Detecting Misinformation and Knowledge Poisoning in Generative AI Systems

nearest to the taste now:
  NEW  0.638 EviReform: Evidence-Guided Query Reformulation for Multi-Hop Graph Retrieva
  kept 0.635 EnSI-RAG: Entity-Structure-Indexed Retrieval-Augmented Generation for Long-
  kept 0.615 Bridging the Question-Answer Gap in Retrieval-Augmented Generation: Hypothe
  NEW  0.612 Coarse Indexing, Fine Evidence: Decoupling Temporal Granularity in Long-Vid
  NEW  0.607 From Retrieved Context to Runtime Control: Adaptive Compression for Edge-ba

dropped out:
   W-RAG: Source-Aware Retrieval for Enterprise Document Generation from Heter
   GTA-RAG: Graph-Trajectory-Augmented Reinforcement Learning for Multi-Turn R


**But it can't ignore what you typed.** The taste vector on its own has a
problem you can already see: ask it for anything and it answers "RAG papers",
because that is all it knows about you. A recommender that ignores the query is
just a list of your old clicks.

So the vector that goes to the store has to be made from both — what you typed
today and what you have liked before. The simplest way to combine two vectors is
a weighted sum, with one dial, α (alpha), saying how much of each:

```
search  =  α · query  +  (1 − α) · taste

query   [0.1, 0.9, 0.2]          taste   [0.866, 0.325, 0.379]

α = 1     [0.1,   0.9,   0.2  ]    all query — the taste vector is multiplied by 0
α = 0.7   [0.33,  0.73,  0.25 ]    0.7·0.1 + 0.3·0.866,  0.7·0.9 + 0.3·0.325,  …
α = 0     [0.866, 0.325, 0.379]    all taste — the query is multiplied by 0
```

Then rescale to length 1, as before. **Before running the sweep below, say what
you expect at each end:** what does α = 1 return, and what does α = 0 return,
for the query *"evaluating hallucination in language models"* with the profile
from the last cell?

In [6]:
from readnext.embed import embed_texts

query_text = "evaluating hallucination in language models"
query_vector = embed_texts([query_text])[0]
taste = olek.taste_vector(vector_of)
judged = set(olek.liked) | set(olek.disliked)

for alpha in [1.0, 0.7, 0.3, 0.0]:
    mixed = alpha * query_vector + (1 - alpha) * taste
    mixed = mixed / np.linalg.norm(mixed)
    print(f"α = {alpha}")
    for score, title in nearest(mixed, skip=judged, k=3):
        print("   ", score, title[:72])

α = 1.0
    0.681 Evaluating Inference-Time Defenses Against Package Hallucination in LLM-
    0.671 Do Large Language Models Hallucinate Electric Fata Morganas?
    0.666 HalluTracer: Hallucination Detection via Depth-Averaging Truth Signals
α = 0.7
    0.665 Do Large Language Models Hallucinate Electric Fata Morganas?
    0.653 Evaluating Inference-Time Defenses Against Package Hallucination in LLM-
    0.652 ReWEIGH the Evidence: Calibrating Token-Level Ordinal Visual Evidence to
α = 0.3
    0.643 From Retrieved Context to Runtime Control: Adaptive Compression for Edge
    0.642 Bridging the Question-Answer Gap in Retrieval-Augmented Generation: Hypo
    0.640 EnSI-RAG: Entity-Structure-Indexed Retrieval-Augmented Generation for Lo
α = 0.0
    0.638 EviReform: Evidence-Guided Query Reformulation for Multi-Hop Graph Retri
    0.635 EnSI-RAG: Entity-Structure-Indexed Retrieval-Augmented Generation for Lo
    0.615 Bridging the Question-Answer Gap in Retrieval-Augmented Generation: Hyp

**Cold start.** Turn 1 for a new user has no clicks, so no taste vector at all —
`taste_vector()` returns `None`, not a vector of zeros. The blend can't be
computed, and the code has to work anyway. The right behaviour is not a special
case, it's the absence of one: with nothing known about you, the search vector
is the query and nothing else, and the result should be *exactly* what book 3's
`search()` would have returned. Not similar — identical.

That's a claim you can test rather than trust. The cell below builds an `Index`
(book 3's store plus the per-paper vectors from earlier, bundled so a ranker can
take one argument), runs `recommend()` for a user with an empty profile, and
asserts it returns the same five papers in the same order as `search()`.

In [7]:
from readnext.config import PipelineConfig
from readnext.recommend import recommend
from readnext.search import build_index

index = build_index(papers, chunks, vectors)
newcomer = Profile(user="newcomer")

from_search = [hit.paper_id for hit in search(store, query_text, k=5)]
from_recommend = [rec.paper.id for rec in recommend(index, query_vector, newcomer, PipelineConfig())]

assert from_search == from_recommend, (from_search, from_recommend)
for paper_id in from_recommend:
    print(by_id[paper_id].title[:80])

Evaluating Inference-Time Defenses Against Package Hallucination in LLM-Generate
Do Large Language Models Hallucinate Electric Fata Morganas?
HalluTracer: Hallucination Detection via Depth-Averaging Truth Signals
ReWEIGH the Evidence: Calibrating Token-Level Ordinal Visual Evidence to Mitigat
Evidence Attribution in Visual Document Understanding without Coordinates or Reg


**The ranking function stays pure.** `recommend()` is two stages, and the α
sweep above is why they are separate. At α = 0 the query vanished *from the
search itself* — the store was asked for "things like my taste" and never heard
about hallucination at all. Personalisation had overwritten relevance.

So the store is only ever asked about the query. Stage 1, `retrieve`, takes the
query vector, pulls `candidate_chunks` chunks, collapses them to papers exactly
as book 3 did, and keeps the top `candidates_k` — forty papers that are *about
what you typed*, the same forty for every user. Stage 2, `rerank`, scores each
of those forty against the blended vector — still by its best chunk, the same
rule as book 3 — and keeps the top `final_k`. Now α only decides the *order* of
papers that are already on topic; it can no longer make the topic disappear.

Every number that stage reads — α, γ, the three k's — lives in one
`PipelineConfig`, so a run is described by one small object rather than by
which arguments happened to be passed. And `recommend()` reads nothing else: no
session, no disk, no globals. Give it the same index, vector, profile and config
and it gives the same answer, which is what lets book 5 call it thousands of
times over logged candidates with a different config each time.

In [8]:
config = PipelineConfig()
print(config, "\n")

for alpha in [1.0, 0.0]:
    print(f"α = {alpha} — same forty candidates, different order")
    for rec in recommend(index, query_vector, olek, PipelineConfig(alpha=alpha))[:3]:
        print(f"    {rec.score:.3f}  {rec.paper.title[:70]}")

top = recommend(index, query_vector, olek, config)[0]
type(top).__name__, top.paper.title, round(top.score, 3), top.why, top.citations

PipelineConfig(use_structured_query=False, use_bm25=False, use_rerank=False, use_mmr=False, fusion='rrf', alpha=0.7, gamma=0.5, candidate_chunks=200, candidates_k=40, final_k=5) 

α = 1.0 — same forty candidates, different order
    0.681  Evaluating Inference-Time Defenses Against Package Hallucination in LL
    0.671  Do Large Language Models Hallucinate Electric Fata Morganas?
    0.666  HalluTracer: Hallucination Detection via Depth-Averaging Truth Signals
α = 0.0 — same forty candidates, different order
    0.509  Do Large Language Models Play Six Degrees of Separation? Measuring Top
    0.503  Dissecting Neuro-Symbolic Quality Assurance for Synthetic Oncology Dat
    0.489  Towards Clinically Faithful Medical Image Captioning via Enhanced Visi


('Recommendation',
 'Do Large Language Models Hallucinate Electric Fata Morganas?',
 0.665,
 '',
 [])

**Never show the same paper twice.** Ask the same question twice and
`recommend()` gives the same five papers, because it has no memory of having
shown them — and it shouldn't. Which papers you have already seen is a fact
about *this sitting*, not about the query or the corpus, and a function that
remembered it would stop being the same function on every call.

So the memory lives elsewhere — a *seen-set*, a plain set of paper ids — and is
handed in on each call as `exclude`. `retrieve` drops those ids before it cuts
the pool to forty, so a seen paper never uses up a slot; the pool is still forty
papers you haven't been shown. The second call below should share nothing with
the first.

In [9]:
first = recommend(index, query_vector, olek, config)
seen = frozenset(rec.paper.id for rec in first)

second = recommend(index, query_vector, olek, config, exclude=seen)
assert not seen & {rec.paper.id for rec in second}

for label, recs in [("first five", first), ("next five", second)]:
    print(label)
    for rec in recs:
        print(f"    {rec.score:.3f}  {rec.paper.title[:70]}")

first five
    0.665  Do Large Language Models Hallucinate Electric Fata Morganas?
    0.653  Evaluating Inference-Time Defenses Against Package Hallucination in LL
    0.652  ReWEIGH the Evidence: Calibrating Token-Level Ordinal Visual Evidence 
    0.651  HalluTracer: Hallucination Detection via Depth-Averaging Truth Signals
    0.649  Do Large Language Models Play Six Degrees of Separation? Measuring Top
next five
    0.637  Evidence Attribution in Visual Document Understanding without Coordina
    0.592  Evaluation Awareness in Language Models: Representation, Verbalization
    0.585  BioMed-Agent-RL: A Meta Learning, All You Need for Biomedical Applicat
    0.584  PEA-DPO: Perception-Enhanced Alignment Direct Preference Optimization 
    0.577  Do LLM Recommenders Know When They're Hallucinating? Auditing Confiden


**The session holds what the ranker won't.** Everything `recommend()` refused
to remember has to live somewhere: whose profile this is, which papers have
been shown, how many questions have been asked, and — shortly — the output
area the cards are drawn into. That somewhere is a `Session`, the one stateful
object in the app.

`ask()` runs the two stages itself rather than calling `recommend()`, because
it needs both halves separately: the forty candidates from stage 1 go into the
log, the five from stage 2 go onto the screen. It adds the five to the seen-set
so the next question can't repeat them, and writes one row to
`data/events.jsonl` — more on that row in a moment. The profile is `olek` from
the last cells, so this user already has a taste; the first click saves it to
`data/profiles/olek.json`, and later sessions load it from there.

In [10]:
from readnext.session import Session

session = Session(index=index, profile=olek)
recs = session.ask("papers on fine-tuning")

print(f"session {session.id}, turn {session.turn}, {len(session.candidates)} candidates, {len(session.seen)} seen")
for rec in recs:
    print(f"    {rec.score:.3f}  {rec.paper.title[:70]}")

session sb6d6e0, turn 1, 40 candidates, 5 seen
    0.611  FiRE: Enhancing MLLMs with Fine-Grained Context Learning for Complex I
    0.564  On the Threat Model of Weird Generalization and Emergent Misalignment
    0.555  What Does CLIP Learn for Regional Geolocalization? Probing Visual Cues
    0.536  CausalCache: Conditional High-Fidelity Restoration for Long-Horizon GU
    0.535  How Architecture and Training Affect TPC Representations Across Experi


**Now click — and now type.** The cards from the start of the book get their
buttons back. Each button's handler is the callback from cell 4 with a real
job: it calls `session.feedback(paper_id, signal)`, which records the click on
the profile, in the session, and on the log row of the turn that showed the
paper, then lights the button you pressed. That is all it does. The list does
not move.

That's deliberate. A click could re-rank the five on the spot — the taste
vector has moved, after all — but then the screen would stop matching the row
just written to the log, and a second click would land on a paper at a position
the row doesn't know about. So a turn is one query, one list, and any number of
judgements against that same list. The taste vector is read once, by `ask()`,
which means what you press here shapes the *next* list.

Above the cards is a text box. It's the same idea as a button, on a different
event: pressing Enter in it calls `session.ask(text)` with what you typed — a
new turn, new candidates, a new row in the log, and the five slots refilled
with fresh buttons.

The session owns one `VBox`, `session.view`: the text box, an output area for
errors, and five card slots built once when the session is created. Display it
once; from then on it updates itself. No cell re-runs. **Try this:** rate two
or three cards, then type a paraphrase of the same question. The pool is
almost the same; the order isn't.

In [11]:
session.view

**Everything you just did is on disk.** Each `ask()` appended one line to
`data/events.jsonl`, and each click rewrote that line's `signals`. A row is an
`Event`, and every field is there for a reader that doesn't exist yet:

- `session`, `turn`, `user` — which sitting, which question in it, whose taste.
- `query` — the exact text you typed. `resolved_query` is empty until book 6
  puts an LLM in front of it.
- `config` — the `PipelineConfig` that produced the row, so a result can always
  be traced to the settings behind it.
- `candidates` — the forty paper ids stage 1 retrieved, in order.
- `shown` — the five that reached the screen, in the order they were first
  drawn.
- `signals` — paper id → `like` / `reading` / `dislike`, filled in as you
  clicked. Empty means you searched and pressed nothing, which is itself a
  fact worth keeping.
- `ts` — when.

Read the last row back and check that it matches what you saw. A click landed
after another question was asked still goes to the row of the turn that showed
the paper, not the latest row.

In [12]:
from readnext import events

last = events.load()[-1]
print(f"{last.session} turn {last.turn} · {last.user} · {last.query!r} · {last.ts}")
print(f"{len(last.candidates)} candidates, {len(last.shown)} shown, signals: {last.signals}")
print("config:", last.config)

sb6d6e0 turn 1 · olek · 'papers on fine-tuning' · 2026-09-13T07:53:45+00:00
40 candidates, 5 shown, signals: {}
config: {'use_structured_query': False, 'use_bm25': False, 'use_rerank': False, 'use_mmr': False, 'fusion': 'rrf', 'alpha': 0.7, 'gamma': 0.5, 'candidate_chunks': 200, 'candidates_k': 40, 'final_k': 5}


**Why the whole candidate pool, not just the five.** `shown` is what you saw;
`candidates` is what you *could* have seen. Keeping the forty is what makes the
row useful tomorrow. Book 5 will take a logged row, run a different stage 2 over
its stored `candidates` — a new α, a reranker, a diversity pass — and ask a
plain question: where did the papers you clicked end up? Near the top, that
config is better for you; further down, worse. No one has to be present, no one
has to label anything, and the same row can be re-asked for every config ever
written.

A row that kept only `shown` can't do that. There's nothing to re-rank — five
papers are five papers — so the click in it could only ever say "the config that
day put this at position 3", and that is all it could ever say. Storing the pool
costs forty ids per search; not storing it makes the row unreplayable forever.

**Now go and use it.** The log is the test set, and it only fills up by being
used. Five or six real sessions is enough for book 5 to have something to
measure; more is better. Vary the questions — a paraphrase of something you've
already asked, a bare acronym like *RLHF* or *RAG*, a topic with a restriction
in it (*"long context, but not for code"*), something you actually want to read
this week. Click honestly: 👍 for an abstract that interests you, 📖 for one
you'd open, 👎 for one you'd rather not have been shown. Skipping a whole list
is a legitimate answer too. Remember that a click shows up on the *next*
question, not this one — rate, then ask again.

Nothing to mark, nothing to judge, nothing to write down — the click is the
label. The cell below is a fresh session with nothing asked yet: type into the
box. It loads your profile from disk, so what you clicked last time already
shapes what you see this time. Re-run the cell whenever you want a new session
id in the log — one sitting, one session.

In [13]:
session = Session(index=index, profile=Profile.load("olek"))
session.view

**What you can do now.** You can ask the app a question, get five papers you
can read and open, react to each one, and ask again and get a list that is
ranked differently because of what you pressed — without re-running anything.
You never see a repeat within a session. And every one of those searches is a
row on disk with the whole candidate pool behind it and every click against
exactly the list you saw.

What you also have is an *opinion*: the recommendations feel good, or they
don't, and you can probably say why in a sentence — "it kept showing me
security papers after one dislike", "the taste vector overwhelms a new topic".
That sentence is book 5's starting point. What book 5 needs and this book
doesn't have is a number: a way to turn "feels better" into a column in a table,
computed from these rows and nothing else.